# Template 06: SHAP DataFrame Creation

**Purpose:** Compute SHAP values and create SHAP dataframes

**Inputs:**
- models/xgb_model.json
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet

**Outputs:**
- results/06_shap_train.parquet
- results/06_shap_test.parquet

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_liab/v1"


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 06: SHAP DATAFRAME CREATION")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 06: SHAP DATAFRAME CREATION
########################################


In [4]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")
print(f"SHAP: {shap.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print("=" * 50)
print()

Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
XGBoost: 3.1.1
SHAP: 0.51.0
Pandas: 3.0.5
NumPy: 2.4.6



In [5]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
target = cfg['experiment']['target']

In [6]:
# Load ENCODED data from Stage 04c
X_train = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
X_test = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

print(f"\n* Train encoded: {X_train.shape}")
print(f"* Test encoded: {X_test.shape}")


* Train encoded: (7481727, 198)
* Test encoded: (7483698, 198)


In [7]:
# Load model as XGBRegressor
model_file = f"{output_base}/models/05c_model.json"
model = xgb.XGBRegressor()
model.load_model(model_file)
print(f"\n* Model loaded: {model_file}")

# Validate features match
from model_validation import validate_features_match
validate_features_match(model, X_train, 'X_train')


* Model loaded: output/car_liab/v1/models/05c_model.json

* Feature Validation:
  Model expects: 198 features
  X_train has: 198 features
  ⚠ Model has no feature names stored (can't verify names)
  ✓ Validation passed



True

In [8]:
# Create SHAP explainer (using new API for XGBoost 3.x compatibility)
print(f"\n* Creating SHAP explainer...")
explainer = shap.Explainer(model)
print(f"  Explainer created")


* Creating SHAP explainer...
  Explainer created


In [9]:
# Compute SHAP values for train (sample if too large)
print(f"\n* Computing SHAP values for train...")
sample_size = min(10000, len(X_train))
X_train_sample = X_train.sample(n=sample_size, random_state=42)

shap_explanation_train = explainer(X_train_sample)
shap_values_train = shap_explanation_train.values
print(f"  SHAP values shape: {shap_values_train.shape}")

# Create SHAP dataframe
shap_df_train = pd.DataFrame(shap_values_train, columns=X_train.columns, index=X_train_sample.index)
shap_df_train['base_value'] = shap_explanation_train.base_values[0] if hasattr(shap_explanation_train.base_values, '__len__') else shap_explanation_train.base_values
print(f"  SHAP dataframe created: {shap_df_train.shape}")


* Computing SHAP values for train...
  SHAP values shape: (10000, 198)
  SHAP dataframe created: (10000, 199)


In [10]:
# Compute SHAP values for test
print(f"\n* Computing SHAP values for test...")
shap_explanation_test = explainer(X_test)
shap_values_test = shap_explanation_test.values
print(f"  SHAP values shape: {shap_values_test.shape}")

# Create SHAP dataframe
shap_df_test = pd.DataFrame(shap_values_test, columns=X_train.columns, index=X_test.index)
shap_df_test['base_value'] = shap_explanation_test.base_values[0] if hasattr(shap_explanation_test.base_values, '__len__') else shap_explanation_test.base_values
print(f"  SHAP dataframe created: {shap_df_test.shape}")


* Computing SHAP values for test...


  SHAP values shape: (7483698, 198)


  SHAP dataframe created: (7483698, 199)


In [11]:
# Save SHAP dataframes
shap_train_file = f"{output_base}/results/06_shap_train.parquet"
shap_test_file = f"{output_base}/results/06_shap_test.parquet"

shap_df_train.to_parquet(shap_train_file)
shap_df_test.to_parquet(shap_test_file)

print(f"\n* Saved:")
print(f"  {shap_train_file}")
print(f"  {shap_test_file}")


* Saved:
  output/car_liab/v1/results/06_shap_train.parquet
  output/car_liab/v1/results/06_shap_test.parquet


In [12]:
# Feature importance (mean absolute SHAP)
feature_importance = pd.DataFrame({
    'feature': X_train.columns.tolist(),
    'mean_abs_shap': np.abs(shap_values_test).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

importance_file = f"{output_base}/results/06_feature_importance.csv"
feature_importance.to_csv(importance_file, index=False)

print(f"\n* Feature importance saved: {importance_file}")
print(f"\nTop 10 features:")
print(feature_importance.head(10))


* Feature importance saved: output/car_liab/v1/results/06_feature_importance.csv

Top 10 features:
                                  feature  mean_abs_shap
46                             ee_bi_imps       0.368543
52                cef_first_poten_dam_ind       0.109408
6                             married_ind       0.081449
4                   multi_pol_unknown_cal       0.048700
3                       multi_pol_yes_cal       0.029170
90                 vc_backup_camera_raw_0       0.022158
54                    cef_lien_holder_ind       0.009961
86   vc_automatic_emergency_braking_raw_3       0.009316
157                   vc_rear_wiper_raw_0       0.009248
92                 vc_backup_camera_raw_5       0.008471


In [13]:
print("\n########################################")
print("# STAGE 06: COMPLETE")
print("########################################")


########################################
# STAGE 06: COMPLETE
########################################
